# MP5: Training Your Diffusion Model!

## Setup environment

In [4]:
# Import essential modules. Feel free to add whatever you need.
import matplotlib.pyplot as plt
import torch # added
from torch import optim # added
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor

## Visualization helper function

In [5]:
def visualize_images_with_titles(images: torch.Tensor, column_names: list[str]):
    """
    Visualize images as a grid and title the columns with the provided names.

    Args:
        images: (N, C, H, W) tensor of images, where N is (number of rows * number of columns)
        column_names: List of column names for the titles.

    Example usage:
    visualize_images_with_titles(torch.randn(16, 1, 32, 32), ['1', '2', '3', '4'])
    """
    num_images, num_columns = images.shape[0], len(column_names)
    assert num_images % num_columns == 0, 'Number of images must be a multiple of the number of columns.'

    num_rows = num_images // num_columns
    fig, axes = plt.subplots(num_rows, num_columns, figsize=(num_columns * 1, num_rows * 1))

    for i, ax in enumerate(axes.flat):
        img = images[i].permute(1, 2, 0).cpu().numpy()
        ax.imshow(img, cmap='gray')
        ax.axis('off')
        if i < num_columns:
            ax.set_title(column_names[i % num_columns])

    plt.tight_layout()
    plt.show()


# Part 1: Training a Single-step Denoising UNet


## Implementing Simple and Composed Ops

In [ ]:
class Conv(nn.Module):
    """
    Conv is a convolutional layer that doesn't change the image resolution, only the channel dimension.
    """
    def __init__(self, in_channels: int, out_channels: int):
        self.in_channels = in_channels
        self.out_channels = out_channels
        super().__init__()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Conv2d(3,1,1) + BN (nn.BatchNorm2d()) + GELU (nn.GELU())
        """
        conv_result = nn.Conv2d(in_channels=self.in_channels, out_channels=self.out_channels, kernel_size=1)(x)
        bn_result = nn.BatchNorm2d(num_features=self.out_channels)(conv_result)
        gelu_result = nn.GELU()(bn_result)
        assert gelu_result.shape == (x.shape[0], self.out_channels, x.shape[2], x.shape[3]), \
            f"Expected shape {(x.shape[0], self.out_channels, x.shape[2], x.shape[3])}, but got {gelu_result.shape}"
        return gelu_result


class DownConv(nn.Module):
    """
    DownConv is a convolutional layer that downsamples the tensor by 2.
    D_in, H, W => D_out, H/2, W/2
    """
    def __init__(self, in_channels: int, out_channels: int):
        self.in_channels = in_channels
        self.out_channels = out_channels
        super().__init__()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Conv2d (3,2,1) + BN + GELU"""
        # Conv2d(3,2,1) + BN (nn.BatchNorm2d()) + GELU (nn.GELU())
        conv_result = nn.Conv2d(in_channels=self.in_channels, out_channels=self.out_channels, kernel_size=3, stride=2, padding=1)(x)
        bn_result = nn.BatchNorm2d(num_features=self.out_channels)(conv_result)
        gelu_result = nn.GELU()(bn_result)
        assert gelu_result.shape == (x.shape[0], self.out_channels, x.shape[2] // 2, x.shape[3] // 2), \
            f"Expected shape {(x.shape[0], self.out_channels, x.shape[2] // 2, x.shape[3] // 2)}, but got {gelu_result.shape}"
        return gelu_result


class UpConv(nn.Module):
    """
    UpConv is a convolutional layer that upsamples the tensor by 2.
    """
    def __init__(self, in_channels: int, out_channels: int):
        self.in_channels = in_channels
        self.out_channels = out_channels
        super().__init__()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """ConTranspose2d (4,2,1) + BN + GELU"""
        # ConvTranspose2d(4,2,1) + BN (nn.BatchNorm2d()) + GELU (nn.GELU())
        conv_result = nn.ConvTranspose2d(in_channels=self.in_channels, out_channels=self.out_channels, kernel_size=4, stride=2, padding=1)(x)
        bn_result = nn.BatchNorm2d(num_features=self.out_channels)(conv_result)
        gelu_result = nn.GELU()(bn_result)
        assert gelu_result.shape == (x.shape[0], self.out_channels, x.shape[2] * 2, x.shape[3] * 2), \
            f"Expected shape {(x.shape[0], self.out_channels, x.shape[2] * 2, x.shape[3] * 2)}, but got {gelu_result.shape}"
        return gelu_result


class Flatten(nn.Module):
    """
    Flatten is an average pooling layer that flattens a 7x7 tensor into a 1x1 tensor.
    7 is the resulting height and width after the downsampling operations.
    """

    def __init__(self):
        super().__init__()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """AvgPool (7) + GELU"""
        # AvgPool(7) + GELU (nn.GELU())
        avg_pool_result = F.avg_pool2d(x, kernel_size=7)
        gelu_result = nn.GELU()(avg_pool_result)
        assert gelu_result.shape == (x.shape[0], x.shape[1], 1, 1), 'GELU output shape is incorrect'
        return gelu_result


class Unflatten(nn.Module):
    """
    Unflatten is a convolutional layer that unflattens/upsamples a 1x1 tensor into a 7x7 tensor.
    """
    def __init__(self, in_channels: int):
        self.in_channels = in_channels
        super().__init__()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """ConvTranspose2d(7,7,0) + BN + GELU"""
        # ConvTranspose2d(7,7,0) + BN (nn.BatchNorm2d()) + GELU (nn.GELU())
        conv_result = nn.ConvTranspose2d(in_channels=self.in_channels, out_channels=self.in_channels, kernel_size=7, stride=1, padding=0)(x)
        bn_result = nn.BatchNorm2d(num_features=self.in_channels)(conv_result)
        gelu_result = nn.GELU()(bn_result)
        assert gelu_result.shape == (x.shape[0], self.in_channels, 7, 7), \
            f"Expected shape {(x.shape[0], self.in_channels, 7, 7)}, but got {gelu_result.shape}"
        return gelu_result


class ConvBlock(nn.Module):
    """
    similar to Conv but includes an additional Conv. 
    Note that it has the same input and output shape as (1) Conv.
    """
    def __init__(self, in_channels: int, out_channels: int):
        self.in_channels = in_channels
        self.out_channels = out_channels
        super().__init__()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Conv2d(3,1,1) + BN (nn.BatchNorm2d()) + GELU (nn.GELU()) + Conv2d(1,1,1) + BN (nn.BatchNorm2d()) + GELU (nn.GELU())
        """
        conv_result = nn.Conv2d(in_channels=self.in_channels, out_channels=self.out_channels, kernel_size=1)(x)
        bn_result = nn.BatchNorm2d(num_features=self.out_channels)(conv_result)
        gelu_result = nn.GELU()(bn_result)
        conv_result_2 = nn.Conv2d(in_channels=self.out_channels, out_channels=self.out_channels, kernel_size=1)(gelu_result)
        bn_result_2 = nn.BatchNorm2d(num_features=self.out_channels)(conv_result_2)
        gelu_result_2 = nn.GELU()(bn_result_2)

        # Check if the output shape is as expected
        assert gelu_result_2.shape == (x.shape[0], self.out_channels, x.shape[2], x.shape[3]), \
            f"Expected shape {(x.shape[0], self.out_channels, x.shape[2], x.shape[3])}, but got {gelu_result_2.shape}"
        return gelu_result_2


class DownBlock(nn.Module):
    """
    similar to DownConv but includes an additional ConvBlock. 
    Note that it has the same input and output shape as (2) DownConv.
    """
    def __init__(self, in_channels: int, out_channels: int):
        self.in_channels = in_channels
        self.out_channels = out_channels
        super().__init__()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """DownConv + ConvBlock"""
        # DownConv + ConvBlock
        downconv_result = DownConv(self.in_channels, self.out_channels)(x)
        convblock_result = ConvBlock(self.out_channels, self.out_channels)(downconv_result)

        # Check if the output shape is as expected      
        assert convblock_result.shape == (x.shape[0], self.out_channels, x.shape[2] // 2, x.shape[3] // 2), \
            f"Expected shape {(x.shape[0], self.out_channels, x.shape[2] // 2, x.shape[3] // 2)}, but got {convblock_result.shape}"
        
        return convblock_result


class UpBlock(nn.Module):
    """
    similar to UpConv but includes an additional ConvBlock. 
    Note that it has the same input and output shape as (3) UpConv.
    """
    def __init__(self, in_channels: int, out_channels: int):
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.up_conv = UpConv(self.in_channels, self.out_channels)
        self.conv_block = ConvBlock(self.out_channels, self.out_channels)
        super().__init__()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """UpConv + ConvBlock"""
        # UpConv + ConvBlock
        upconv_result = self.up_conv(x)
        convblock_result = self.conv_block(upconv_result)

        # Check if the output shape is as expected
        assert convblock_result.shape == (x.shape[0], self.out_channels, x.shape[2] * 2, x.shape[3] * 2), \
            f"Expected shape {(x.shape[0], self.out_channels, x.shape[2] * 2, x.shape[3] * 2)}, but got {convblock_result.shape}"
        return convblock_result

## Implementing Unconditional UNet

In [9]:
class UnconditionalUNet(nn.Module):
    def __init__(
        self,
        in_channels: int,
        num_hiddens: int,
    ):
        self.in_channels = in_channels
        self.num_hiddens = num_hiddens
        super().__init__()

        # Layers in U-Net
        self.first_conv = ConvBlock(self.in_channels, self.num_hiddens)

        # Downsampling Path
        self.down1 = DownBlock(self.num_hiddens, self.num_hiddens)
        self.down2 = DownBlock(self.num_hiddens, self.num_hiddens * 2)

        # Flatten and Unflatten (bottom of U-Net)
        self.flatten = Flatten()
        self.unflatten = Unflatten(self.num_hiddens * 2)

        # Upsampling Path
        # upsample1 is the one that receives the concatenatino of Unflatten and DownBlock
        self.upsample1 = UpBlock(self.num_hiddens * 4, self.num_hiddens)
        self.upsample2 = UpBlock(self.num_hiddens * 2, self.num_hiddens)

        # Final Convolution Blocks
        self.final_conv1 = ConvBlock(self.num_hiddens * 2, self.num_hiddens)
        self.final_conv2 = nn.Conv2d(self.num_hiddens, self.in_channels, kernel_size=3, stride=1, padding=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        assert x.shape[-2:] == (28, 28), "Expect input shape to be (28, 28)."

        # Encoder?
        c1 = self.initial_conv(x)
        d1 = self.down1(c1)
        d2 = self.down2(d1)

        # Bottleneck
        flat = self.flatten(d2)
        b = self.unflatten(flat)

        # Decoder
        cat1 = torch.cat((b, d2), dim=1)
        u1 = self.upsample1(cat1)

        ## Second level
        cat2 = torch.cat((u1, d1), dim=1)
        u2 = self.upsample2(cat2)

        ## Final (top) level
        cat3 = torch.cat((u2, c1), dim=1)
        f1 = self.final_conv1(cat3)

        # Final convolution
        out_res = self.final_conv2(f1)

        assert out_res.shape == x.shape, f"Output shape {out_res.shape} doesn't match input shape {x.shape}"
        
        return out_res

## Visualizing the noising process

In [5]:
dataset = MNIST(root="data", download=True, transform=ToTensor(), train=True)

# TODO

100.0%
100.0%
100.0%
100.0%


## Training a Single-Step Unconditional UNet

- Plot the loss curve
- Sample results on the test set

In [6]:
dataset = MNIST(root='data', download=True, transform=ToTensor(), train=True)
dataloader = DataLoader(dataset, batch_size=256, shuffle=True)

# TODO

## Out-of-Distribution Testing

In [ ]:
# TODO

# Part 2: Training a Diffusion Model

## Implementing a Time-conditioned UNet

In [7]:
class FCBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError()


class TimeConditionalUNet(nn.Module):
    def __init__(
        self,
        in_channels: int,
        num_classes: int,
        num_hiddens: int,
    ):
        super().__init__()

    def forward(
        self,
        x: torch.Tensor,
        t: torch.Tensor,
    ) -> torch.Tensor:
        """
        Args:
            x: (N, C, H, W) input tensor.
            t: (N,) normalized time tensor.

        Returns:
            (N, C, H, W) output tensor.
        """
        assert x.shape[-2:] == (28, 28), "Expect input shape to be (28, 28)."
        raise NotImplementedError()

## Implementing DDPM Forward and Inverse Process for Time-conditioned Denoising

In [8]:
def ddpm_schedule(beta1: float, beta2: float, num_ts: int) -> dict:
    """Constants for DDPM training and sampling.

    Arguments:
        beta1: float, starting beta value.
        beta2: float, ending beta value.
        num_ts: int, number of timesteps.

    Returns:
        dict with keys:
            betas: linear schedule of betas from beta1 to beta2.
            alphas: 1 - betas.
            alpha_bars: cumulative product of alphas.
    """
    assert beta1 < beta2 < 1.0, "Expect beta1 < beta2 < 1.0."
    raise NotImplementedError()

In [9]:
def ddpm_forward(
    unet: TimeConditionalUNet,
    ddpm_schedule: dict,
    x_0: torch.Tensor,
    num_ts: int,
) -> torch.Tensor:
    """Algorithm 1 of the DDPM paper.

    Args:
        unet: TimeConditionalUNet
        ddpm_schedule: dict
        x_0: (N, C, H, W) input tensor.
        num_ts: int, number of timesteps.
    Returns:
        (,) diffusion loss.
    """
    unet.train()
    # YOUR CODE HERE.
    raise NotImplementedError()

In [10]:
@torch.inference_mode()
def ddpm_sample(
    unet: TimeConditionalUNet,
    ddpm_schedule: dict,
    img_wh: tuple[int, int],
    num_ts: int,
    seed: int = 0,
) -> torch.Tensor:
    """Algorithm 2 of the DDPM paper with classifier-free guidance.

    Args:
        unet: TimeConditionalUNet
        ddpm_schedule: dict
        img_wh: (H, W) output image width and height.
        num_ts: int, number of timesteps.
        seed: int, random seed.

    Returns:
        (N, C, H, W) final sample.
    """
    unet.eval()
    # YOUR CODE HERE.
    raise NotImplementedError()

In [11]:
class DDPM(nn.Module):
    def __init__(
        self,
        unet: TimeConditionalUNet,
        betas: tuple[float, float] = (1e-4, 0.02),
        num_ts: int = 300,
        p_uncond: float = 0.1,
    ):
        super().__init__()
        self.unet = unet
        self.num_ts = num_ts
        self.p_uncond = p_uncond

        self.ddpm_schedule = nn.ParameterDict(ddpm_schedule(betas[0], betas[1], num_ts))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, C, H, W) input tensor.

        Returns:
            (,) diffusion loss.
        """
        return ddpm_forward(
            self.unet, self.ddpm_schedule, x, self.num_ts
        )

    @torch.inference_mode()
    def sample(
        self,
        img_wh: tuple[int, int],
        seed: int = 0,
    ):
        return ddpm_sample(
            self.unet, self.ddpm_schedule, img_wh, self.num_ts, seed
        )

## Training the Time-conditioned UNet

- Plot the loss curve
- Sample results on the test set

In [12]:
dataset = MNIST(root='data', download=True, transform=ToTensor(), train=True)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

# TODO

### Implementing class-conditioned UNet

In [13]:
class ClassConditionalUNet(nn.Module):
    def __init__(
        self,
        in_channels: int,
        num_classes: int,
        num_hiddens: int,
    ):
        super().__init__()

    def forward(
        self,
        x: torch.Tensor,
        c: torch.Tensor,
        t: torch.Tensor,
        mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        """
        Args:
            x: (N, C, H, W) input tensor.
            c: (N,) int64 condition tensor.
            t: (N,) normalized time tensor.
            mask: (N,) mask tensor. If not None, mask out condition when mask == 0.

        Returns:
            (N, C, H, W) output tensor.
        """
        assert x.shape[-2:] == (28, 28), "Expect input shape to be (28, 28)."
        raise NotImplementedError()

In [15]:
def ddpm_forward(
    unet: ClassConditionalUNet,
    ddpm_schedule: dict,
    x_0: torch.Tensor,
    c: torch.Tensor,
    p_uncond: float,
    num_ts: int,
) -> torch.Tensor:
    """Algorithm 1 of the DDPM paper.

    Args:
        unet: ClassConditionalUNet
        ddpm_schedule: dict
        x_0: (N, C, H, W) input tensor.
        c: (N,) int64 condition tensor.
        p_uncond: float, probability of unconditioning the condition.
        num_ts: int, number of timesteps.

    Returns:
        (,) diffusion loss.
    """
    unet.train()
    # YOUR CODE HERE.
    raise NotImplementedError()

In [16]:
@torch.inference_mode()
def ddpm_sample(
    unet: ClassConditionalUNet,
    ddpm_schedule: dict,
    c: torch.Tensor,
    img_wh: tuple[int, int],
    num_ts: int,
    guidance_scale: float = 5.0,
    seed: int = 0,
) -> torch.Tensor:
    """Algorithm 2 of the DDPM paper with classifier-free guidance.

    Args:
        unet: ClassConditionalUNet
        ddpm_schedule: dict
        c: (N,) int64 condition tensor. Only for class-conditional
        img_wh: (H, W) output image width and height.
        num_ts: int, number of timesteps.
        guidance_scale: float, CFG scale.
        seed: int, random seed.

    Returns:
        (N, C, H, W) final sample.
        (N, T_animation, C, H, W) caches.
    """
    unet.eval()
    # YOUR CODE HERE.
    raise NotImplementedError()

In [17]:
class DDPM(nn.Module):
    def __init__(
        self,
        unet: ClassConditionalUNet,
        betas: tuple[float, float] = (1e-4, 0.02),
        num_ts: int = 300,
        p_uncond: float = 0.1,
    ):
        super().__init__()
        self.unet = unet
        self.betas = betas
        self.num_ts = num_ts
        self.p_uncond = p_uncond
        self.ddpm_schedule = nn.ParameterDict(ddpm_schedule(betas[0], betas[1], num_ts))

    def forward(self, x: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, C, H, W) input tensor.
            c: (N,) int64 condition tensor.

        Returns:
            (,) diffusion loss.
        """
        return ddpm_forward(
            self.unet, self.ddpm_schedule, x, c, self.p_uncond, self.num_ts
        )

    @torch.inference_mode()
    def sample(
        self,
        c: torch.Tensor,
        img_wh: tuple[int, int],
        guidance_scale: float = 5.0,
        seed: int = 0,
    ):
        return ddpm_sample(
            self.unet, self.ddpm_schedule, c, img_wh, self.num_ts, guidance_scale, seed
        )

## Training the Class-conditioned UNet

- Plot the loss curve
- Sample results on the test set

In [18]:
dataset = MNIST(root='data', download=True, transform=ToTensor(), train=True)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

# TODO